# Extract YouTube Transcripts

- **Tasks:**
    
    1. Find any YouTube clip where predictions exists. For ex, [Immediate 2025 NCAA tournament Final Four and championship picks](https://youtu.be/-rjnvL9LL3U?si=QMFJYAQ8Q85lTNCD). Below, we include all the videos we extract data from.
    2. Use the [`youtube-transcript-api`](https://pypi.org/project/youtube-transcript-api/) to retrieve the transcript.
    3. Extract the transcript snippets.
    4. Save the raw transcript snippets.

In [1]:
import os
import sys

import pandas as pd

from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Preprocessing code

- Will use the below class and functions for both video transcripts.

---

1. `YouTubeTranscriptApi()` to fetch the YT transcripts.
2. `extract_data()` to transform data into Pandas DataFrame.

In [3]:
ytt_api = YouTubeTranscriptApi()

In [4]:
def extract_data(snippets) -> pd.DataFrame:
    """Extract the text, start time, and duration from the YT `FetchedTranscriptSnippet()`."""
    snippet_texts = []
    snippet_starts = []
    snippet_durations = []
    data = {}

    for snippets_idx in range(len(snippets)):
        snippet = snippets[snippets_idx]
        snippet_texts.append(snippet.text)
        snippet_starts.append(snippet.start)
        snippet_durations.append(snippet.duration)

    data['Text'] = snippet_texts
    data['Start Time'] = snippet_starts
    data['Duration'] = snippet_durations

    df = pd.DataFrame(data=data)
    return df

## Transcripts

| # | Title | Link | Video ID |
|---|---|---|---|
| 1 | Immediate 2025 NCAA tournament Final Four and championship picks | https://www.youtube.com/watch?v=-rjnvL9LL3U | -rjnvL9LL3U 
| 2 | FULL PREVIEW & PICKS: Patriots vs. Seahawks Super Bowl LX 🏆 Who wins the Lombardi Trophy? \| NFL Live | https://www.youtube.com/watch?v=ZZN7BAYeOtc | ZZN7BAYeOtc |
| 3 | FIRST TAKE'S SUPER BOWL PICKS! The crew is going with... 😱 | https://www.youtube.com/watch?v=mBK8o5orBbE | mBK8o5orBbE |
| 4 | NFL Predictions and Picks For Super Bowl LX [Patriots vs Seahawks] - Best Bets ✅ | https://www.youtube.com/watch?v=LXPQrZV4Cfw | LXPQrZV4Cfw |
| 5 | Rich Eisen’s Pick to Win the Seahawks vs Patriots Super Bowl LX Is….? - The Rich Eisen Show | https://www.youtube.com/watch?v=fUmJAtFEGn8 | fUmJAtFEGn8 |
| 6 | The Pat McAfee Show's Picks For Super Bowl LX | https://www.youtube.com/watch?v=MTVAkVkkaz4 | MTVAkVkkaz4 |
| 7 | Super Bowl LX On-Site Preview: Picks, Predictions, Everything you need to know for Patriots-Seahawks | https://www.youtube.com/watch?v=Z0xP3GNpjkw | Z0xP3GNpjkw |
| 8 | Why Stephen A. IS NOT all in on the Boston Celtics 😳 --- First Take | https://www.youtube.com/watch?v=QmxK_3zhLzQ | QmxK_3zhLzQ |

In [5]:
video_id = "QmxK_3zhLzQ"
transcript_snippets = ytt_api.fetch(video_id)
transcript_snippets[:7]

[FetchedTranscriptSnippet(text="Did he put on a show last night? We'll", start=0.08, duration=2.64),
 FetchedTranscriptSnippet(text='dive into that in a minute. Good morning', start=1.439, duration=2.801),
 FetchedTranscriptSnippet(text='everyone. Welcome to First Take. I am', start=2.72, duration=4.079),
 FetchedTranscriptSnippet(text='Shay Cornet. Stephen A. Smith. Oh, and', start=4.24, duration=4.319),
 FetchedTranscriptSnippet(text='good morning, Big Perk. How are you this', start=6.799, duration=3.361),
 FetchedTranscriptSnippet(text='morning?', start=8.559, duration=5.761),
 FetchedTranscriptSnippet(text='>> Good morning. Good morning. Hold up.', start=10.16, duration=6.08)]

In [6]:
snippets_df = extract_data(transcript_snippets)
snippets_df

,Text,Start Time,Duration
0,Did he put on a show last night? We'll,0.080,2.640
1,dive into that in a minute. Good morning,1.439,2.801
2,everyone. Welcome to First Take. I am,2.720,4.079
3,"Shay Cornet. Stephen A. Smith. Oh, and",4.240,4.319
4,"good morning, Big Perk. How are you this",6.799,3.361
...,...,...,...
251,he's busting everybody's ass. That's,531.519,3.841
252,why.,534.000,3.120
253,>> You got JB Bickerstaff saying something.,535.360,3.840
254,Mike Brown saying something. And now,537.120,6.120


In [7]:
snippets_df['Video ID'] = video_id
snippets_df

,Text,Start Time,Duration,Video ID
0,Did he put on a show last night? We'll,0.080,2.640,QmxK_3zhLzQ
1,dive into that in a minute. Good morning,1.439,2.801,QmxK_3zhLzQ
2,everyone. Welcome to First Take. I am,2.720,4.079,QmxK_3zhLzQ
3,"Shay Cornet. Stephen A. Smith. Oh, and",4.240,4.319,QmxK_3zhLzQ
4,"good morning, Big Perk. How are you this",6.799,3.361,QmxK_3zhLzQ
...,...,...,...,...
251,he's busting everybody's ass. That's,531.519,3.841,QmxK_3zhLzQ
252,why.,534.000,3.120,QmxK_3zhLzQ
253,>> You got JB Bickerstaff saying something.,535.360,3.840,QmxK_3zhLzQ
254,Mike Brown saying something. And now,537.120,6.120,QmxK_3zhLzQ


In [8]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "raw_transcripts")
DataProcessing.save_to_file(snippets_df, path=save_data_path, prefix=f'{video_id}', save_file_type='csv', include_version=False)

Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/prediction_acquition-youtube/../data/yt/raw_transcripts/QmxK_3zhLzQ.csv
